<a href="https://colab.research.google.com/github/A-Kuo/Language-Model-Hallucination-Detection-via-Entropy-Divergence/blob/main/colab/ablation_study.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# AED Ablation Study
## Which Signal Matters? Entropy vs KL Divergence

**Runtime:** ~8 min on T4 GPU  
**Before you run:** Runtime → Change runtime type → T4 GPU

In [ ]:
#@title 1. Setup — Clone repo, install deps, verify GPU
import os, sys, shutil

# ALWAYS reset to /content to prevent nested chdir
os.chdir('/content')

# Fresh clone every time
REPO = 'Language-Model-Hallucination-Detection-via-Entropy-Divergence'
if os.path.exists(REPO):
    shutil.rmtree(REPO)
    print('Removed old clone')

!git clone --depth 1 https://github.com/A-Kuo/{REPO}.git

# Absolute paths used everywhere
REPO_ROOT = os.path.join('/content', REPO)
V1_DIR = os.path.join(REPO_ROOT, 'v1')
RESULTS_DIR = os.path.join(REPO_ROOT, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

if V1_DIR not in sys.path:
    sys.path.insert(0, V1_DIR)

print(f'REPO_ROOT: {REPO_ROOT}')

# Install deps
!pip install -q transformers datasets accelerate scikit-learn

import torch
if not torch.cuda.is_available():
    print('\n*** NO GPU — go to Runtime > Change runtime type > T4 GPU ***')
else:
    print(f'\nGPU: {torch.cuda.get_device_name(0)}')

print('\n=== Setup complete ===')

In [ ]:
#@title 2. Load model and dataset
import os, time, warnings
import numpy as np
import torch
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
warnings.filterwarnings('ignore', message='divide by zero')
warnings.filterwarnings('ignore', message='invalid value')

from transformers import AutoModelForCausalLM, AutoTokenizer
from run_experiment import load_halueval_qa, extract_features, BiLSTMClassifier, auroc, fpr_at_tpr

print('Loading Pythia-160m...')
tokenizer = AutoTokenizer.from_pretrained('EleutherAI/pythia-160m')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained('EleutherAI/pythia-160m', output_attentions=True)
model.eval().cuda()
num_layers = model.config.num_hidden_layers
print(f'Model: {num_layers} layers')

print('\nLoading HaluEval QA (200 samples)...')
pairs = load_halueval_qa(max_samples=200)
print(f'Loaded {len(pairs)} pairs')

In [ ]:
#@title 3. Extract features
features, labels = [], []
for i, (ctx, cont, label) in enumerate(pairs):
    feat = extract_features(model, tokenizer, ctx, cont, 'cuda')
    feat = np.nan_to_num(feat, nan=0.0, posinf=0.0, neginf=0.0)
    features.append(feat)
    labels.append(label)
    if (i + 1) % 50 == 0:
        print(f'  {i+1}/{len(pairs)} done')

X = np.array(features)
y = np.array(labels)
num_layers = X.shape[1] // 2
print(f'\nFeature matrix: {X.shape} ({num_layers} layers)')

In [ ]:
#@title 4. Run ablation — Entropy vs KL vs Both
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

ablations = [
    ('Entropy only',    list(range(num_layers))),
    ('KL only',         list(range(num_layers, num_layers * 2))),
    ('Both (full AED)', list(range(num_layers * 2))),
]

results = {}
print('Running ablation study...\n')

for name, cols in ablations:
    Xtr = X_train[:, cols]
    Xte = X_test[:, cols]
    clf = BiLSTMClassifier(input_dim=len(cols))
    clf.fit(Xtr, y_train.astype(np.float32))
    proba = clf.predict_proba(Xte)
    auc = auroc(y_test, proba)
    fpr = fpr_at_tpr(y_test, proba)
    results[name] = {'auroc': round(auc, 4), 'fpr90': round(fpr, 4)}
    print('{:20} AUROC={:.4f}  FPR@90%TPR={:.4f}'.format(name, auc, fpr))

print('\n' + '='*60)
print('ABLATION RESULTS')
print('='*60)

In [ ]:
#@title 5. Save results and generate LaTeX table
import json

baseline_auc = results['Both (full AED)']['auroc']

print('LaTeX table rows for paper.tex:\n')
for name, res in results.items():
    delta = res['auroc'] - baseline_auc
    method = name.replace(' ', r'\_')
    print('{:30} & {:.3f} & {:+.3f} {}'.format(method, res['auroc'], delta, r'\\'))

out_file = os.path.join(RESULTS_DIR, 'ablation_results.json')
with open(out_file, 'w') as f:
    json.dump(results, f, indent=2)
print('\nSaved:', out_file)

In [ ]:
#@title 6. Visualization
import matplotlib.pyplot as plt

methods = list(results.keys())
aurocs = [results[m]['auroc'] for m in methods]
fprs = [results[m]['fpr90'] for m in methods]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.barh(methods, aurocs, color=colors)
ax1.set_xlim(0.5, 1.0)
ax1.set_xlabel('AUROC')
ax1.set_title('Ablation: AUROC by Feature Set')
ax1.axvline(x=0.5, color='red', linestyle='--', alpha=0.5)
for i, v in enumerate(aurocs):
    ax1.text(v + 0.01, i, '{:.3f}'.format(v), va='center')

ax2.barh(methods, fprs, color=colors)
ax2.set_xlabel('FPR@90%TPR (lower is better)')
ax2.set_title('Ablation: False Positive Rate')
for i, v in enumerate(fprs):
    ax2.text(v + 0.02, i, '{:.3f}'.format(v), va='center')

plt.tight_layout()
fig_path = os.path.join(RESULTS_DIR, 'figure_ablation_comparison.png')
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()
print('Saved:', fig_path)

In [ ]:
#@title 7. Auto-commit to GitHub (needs GH_TOKEN in Secrets)
import os

GH_TOKEN = os.environ.get('GH_TOKEN')
if not GH_TOKEN:
    try:
        from google.colab import userdata
        GH_TOKEN = userdata.get('GH_TOKEN')
    except Exception:
        pass

if not GH_TOKEN:
    print('No GH_TOKEN — downloading results instead')
    from google.colab import files
    for fname in [out_file, fig_path]:
        if os.path.exists(fname):
            files.download(fname)
            print(f'  Downloaded: {os.path.basename(fname)}')
else:
    os.chdir(REPO_ROOT)
    remote = 'https://{}@github.com/A-Kuo/{}.git'.format(GH_TOKEN, REPO)
    !git remote set-url origin {remote}
    !git config user.email "colab@ablation.ai"
    !git config user.name "Ablation Bot"
    !git add results/
    best = results['Both (full AED)']['auroc']
    !git commit -m "results: ablation study (full AED AUROC={best})" || echo 'Nothing new'
    !git push origin main
    print('\nResults committed to GitHub!')